# 01 · 資料分析全流程 + 評估指標

**對應教學 App**：📊 資料分析基礎、🎯 評估指標

**你會做到**：
1. 載入資料、做 EDA（探索性資料分析）
2. 處理缺值、做特徵工程
3. 切訓練/測試、訓練隨機森林
4. **算出 Recall / Precision / F1、畫混淆矩陣和 ROC 曲線**
5. 調整閾值，看指標怎麼動
6. 看特徵重要性

**怎麼操作**：點一格程式碼，按 `Shift + Enter` 執行。從上往下一格一格按。

---

## 0. 載入套件與設定

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# 讓圖表能顯示中文（Windows）
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Microsoft YaHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("套件載入完成 ✅")

## 1. 產生一份「客戶流失」資料

實務上你會用 `pd.read_csv("data.csv")` 讀真實檔案。
這裡我們自己造一份，好處是**我們知道正確答案是什麼**，方便驗證學到的東西對不對。

情境：一家電信公司想預測**哪些客戶下個月會退租**（流失）。

In [ ]:
def make_churn_data(n=4000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)

    tenure   = rng.integers(1, 73, n)                       # 已經用了幾個月
    monthly  = rng.normal(70, 25, n).clip(20, 150)          # 月費
    support  = rng.poisson(1.2, n)                          # 過去半年客訴次數
    contract = rng.choice(["月租", "一年約", "兩年約"], n, p=[.55, .25, .20])
    payment  = rng.choice(["信用卡", "轉帳", "超商繳費"], n, p=[.45, .35, .20])
    age      = rng.normal(42, 14, n).clip(18, 80).round()

    # 真實的流失規則（現實中你不知道這條式子，模型要自己學出來）
    logit = (
        -1.4
        - 0.045 * tenure                                    # 用越久越不會走
        + 0.020 * monthly                                   # 月費越高越想走
        + 0.55 * support                                    # 客訴越多越想走
        + np.where(contract == "月租", 1.5, 0)              # 月租最容易走
        + np.where(contract == "兩年約", -0.9, 0)
        + np.where(payment == "超商繳費", 0.6, 0)
        - 0.012 * (age - 42)
        + rng.normal(0, 0.6, n)
    )
    churn = (1 / (1 + np.exp(-logit)) > rng.uniform(0, 1, n)).astype(int)

    df = pd.DataFrame({
        "年齡": age, "使用月數": tenure, "月費": monthly.round(1),
        "客訴次數": support, "合約類型": contract, "付款方式": payment,
        "是否流失": churn,
    })

    # 故意弄髒資料：製造缺值和重複列（真實資料一定是髒的）
    df.loc[rng.choice(n, 220, replace=False), "月費"] = np.nan
    df.loc[rng.choice(n, 90, replace=False), "年齡"] = np.nan
    df = pd.concat([df, df.sample(60, random_state=seed)], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = make_churn_data()
df.head(10)

## 2. EDA：先看資料長什麼樣

**任何專案的第一步都是這三行**。不要跳過。

In [ ]:
print("資料形狀 (列數, 欄數):", df.shape)
print()
df.info()

In [ ]:
# 數值欄位的統計摘要
df.describe().T

In [ ]:
# 【最重要】看正負樣本比例 —— 決定了後面要用什麼指標
print(df["是否流失"].value_counts())
print()
print("流失率：", f"{df['是否流失'].mean():.1%}")
print()
print("👉 正例只佔約四分之一，屬於中度不平衡。")
print("👉 如果我們全部猜「不流失」，準確率就有", f"{1 - df['是否流失'].mean():.1%}")
print("   —— 這就是為什麼不能只看 Accuracy。")

In [ ]:
# 看缺值和重複
print("每欄缺值數：")
print(df.isna().sum())
print()
print("重複列數：", df.duplicated().sum())

In [ ]:
# 視覺化：流失 vs 沒流失，各欄位的分布差在哪
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ["使用月數", "月費", "客訴次數"]):
    for v, label, c in [(0, "沒流失", "#2563eb"), (1, "流失", "#dc2626")]:
        ax.hist(df[df["是否流失"] == v][col].dropna(), bins=25, alpha=0.6, label=label, color=c)
    ax.set_title(f"{col} 的分布")
    ax.set_xlabel(col)
    ax.legend()

plt.tight_layout()
plt.show()

print("👉 看得出「使用月數」短的、「客訴次數」多的，流失比例明顯較高。")
print("   這代表這兩個欄位對模型會很有用。")

In [ ]:
# 類別欄位的流失率
for col in ["合約類型", "付款方式"]:
    print(f"=== {col} 的流失率 ===")
    print(df.groupby(col)["是否流失"].agg(["mean", "count"]).round(3))
    print()

## 3. 資料清理

三個標準動作：**刪重複 → 補缺值 → 檢查異常值**。

In [ ]:
df_clean = df.drop_duplicates().copy()
print(f"刪掉 {len(df) - len(df_clean)} 筆重複列")

# 用中位數補（不用平均數，因為中位數不怕極端值）
for col in ["月費", "年齡"]:
    median = df_clean[col].median()
    # 多開一欄記錄「這格原本是缺的」—— 缺值本身可能就是訊號
    df_clean[f"{col}_原本缺值"] = df_clean[col].isna().astype(int)
    df_clean[col] = df_clean[col].fillna(median)
    print(f"{col} 用中位數 {median:.1f} 補值")

print()
print("剩餘缺值：", df_clean.isna().sum().sum())

## 4. 特徵工程

把原始欄位加工成模型好懂的形式。這裡做兩件事：

1. **類別欄位 → One-Hot Encoding**（把文字變成 0/1 欄位）
2. **造新欄位**：`平均每月客訴次數`（這種比值欄位往往比原本兩欄都有用）

In [ ]:
# ① 造新特徵
df_clean["每月客訴率"] = df_clean["客訴次數"] / df_clean["使用月數"]
df_clean["累計消費"] = df_clean["月費"] * df_clean["使用月數"]

# ② One-Hot Encoding
df_model = pd.get_dummies(df_clean, columns=["合約類型", "付款方式"], drop_first=True)

print("特徵工程後的欄位：")
print(list(df_model.columns))
print()
df_model.head()

## 5. 切訓練 / 測試

⚠️ **兩個關鍵**：
- `stratify=y`：確保訓練集和測試集的**流失比例一樣**
- **先切分，再做任何需要 fit 的前處理**（否則會資料洩漏）

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=["是否流失"])
y = df_model["是否流失"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"訓練集：{X_train.shape[0]} 筆　流失率 {y_train.mean():.1%}")
print(f"測試集：{X_test.shape[0]} 筆　流失率 {y_test.mean():.1%}")
print()
print("👉 兩邊比例一致，stratify 生效了 ✅")

## 6. 先建 Baseline（基準線）

**永遠先跑最笨的方法。** 沒有基準線，你不知道模型到底有沒有用。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")   # 全部猜「不流失」
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)

print("=== Baseline：全部猜「不流失」 ===")
print(f"Accuracy : {accuracy_score(y_test, y_dummy):.3f}   ← 看起來很漂亮")
print(f"Recall   : {recall_score(y_test, y_dummy):.3f}   ← 一個流失客戶都沒抓到！")
print(f"Precision: {precision_score(y_test, y_dummy, zero_division=0):.3f}")
print()
print("👉 這就是為什麼 Accuracy 在不平衡資料上沒有意義。")

## 7. 訓練隨機森林

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,          # 種 300 棵樹
    max_depth=10,              # 每棵最多 10 層，防過擬合
    min_samples_leaf=5,        # 葉子至少 5 個樣本，防過擬合
    class_weight="balanced",   # 資料不平衡 → 提高少數類的權重
    random_state=RANDOM_STATE,
    n_jobs=-1,                 # 用全部 CPU 核心
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]     # 注意：算 AUC 要用機率，不是 0/1

print("訓練完成 ✅")

## 8. 【核心】評估指標

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(classification_report(y_test, y_pred, target_names=["沒流失", "流失"], digits=3))
print(f"AUC: {roc_auc_score(y_test, y_proba):.3f}")

In [ ]:
# 手動拆解混淆矩陣，親手算一次指標（確認自己真的懂公式）
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("=== 混淆矩陣 ===")
print(f"TP (真的會走，抓到了)   : {tp}")
print(f"FP (不會走，被誤抓)     : {fp}")
print(f"FN (真的會走，漏掉了)   : {fn}   ← 這些是損失的客戶")
print(f"TN (不會走，正確放過)   : {tn}")
print()
print("=== 親手算 ===")
print(f"Recall    = TP / (TP + FN) = {tp} / {tp + fn} = {tp / (tp + fn):.3f}")
print(f"Precision = TP / (TP + FP) = {tp} / {tp + fp} = {tp / (tp + fp):.3f}")

p, r = tp / (tp + fp), tp / (tp + fn)
print(f"F1        = 2PR / (P + R)  = {2 * p * r / (p + r):.3f}")
print()
print("✅ 對照上面 classification_report 的數字，應該完全一樣。")

In [ ]:
# 畫混淆矩陣
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

labels = [["TN", "FP"], ["FN", "TP"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{labels[i][j]}\n{cm[i, j]}", ha="center", va="center",
                fontsize=16, color="white" if cm[i, j] > cm.max() / 2 else "black")

ax.set_xticks([0, 1], ["預測：沒流失", "預測：流失"])
ax.set_yticks([0, 1], ["實際：沒流失", "實際：流失"])
ax.set_title("混淆矩陣")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 9. 調整閾值：Recall 和 Precision 的拉扯

模型預設用 0.5 當閾值，但**這不是聖旨**。

商業判斷：挽回一個客戶的成本是 300 元，一個客戶的年價值是 6000 元。
**漏掉一個（FN）損失 6000，誤抓一個（FP）只浪費 300。**
→ 所以我們應該**降低閾值，提高 Recall**。

In [ ]:
rows = []
for t in np.arange(0.10, 0.91, 0.05):
    pred_t = (y_proba >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, pred_t).ravel()
    rows.append({
        "閾值": round(t, 2),
        "Recall": recall_score(y_test, pred_t),
        "Precision": precision_score(y_test, pred_t, zero_division=0),
        "F1": f1_score(y_test, pred_t, zero_division=0),
        "漏掉FN": fn_, "誤抓FP": fp_,
        "淨效益": tp_ * 6000 * 0.4 - (tp_ + fp_) * 300,   # 假設挽回成功率 40%
    })

res = pd.DataFrame(rows)
res.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(res["閾值"], res["Recall"], "o-", label="Recall", color="#dc2626", lw=2)
axes[0].plot(res["閾值"], res["Precision"], "s-", label="Precision", color="#2563eb", lw=2)
axes[0].plot(res["閾值"], res["F1"], "^-", label="F1", color="#16a34a", lw=2)
axes[0].axvline(0.5, ls="--", c="gray", label="預設閾值 0.5")
axes[0].set_xlabel("閾值"); axes[0].set_ylabel("分數")
axes[0].set_title("閾值 vs 指標：兩條線永遠反向")
axes[0].legend(); axes[0].grid(alpha=.3)

best = res.loc[res["淨效益"].idxmax()]
axes[1].plot(res["閾值"], res["淨效益"], "o-", color="#d97706", lw=2)
axes[1].axvline(best["閾值"], ls="--", c="#16a34a",
                label=f"最佳閾值 {best['閾值']}")
axes[1].set_xlabel("閾值"); axes[1].set_ylabel("預估淨效益（元）")
axes[1].set_title("用「商業效益」選閾值，而不是用 F1")
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.show()

print(f"👉 最佳閾值 = {best['閾值']}，此時 Recall = {best['Recall']:.1%}、"
      f"Precision = {best['Precision']:.1%}")
print(f"   預估淨效益 {best['淨效益']:,.0f} 元（預設 0.5 時是 "
      f"{res[res['閾值'] == 0.5]['淨效益'].iloc[0]:,.0f} 元）")
print()
print("💡 這就是面試時要講的：把技術指標翻譯成商業語言。")

## 10. ROC 曲線

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

fpr, tpr, _ = roc_curve(y_test, y_proba)
prec, rec, _ = precision_recall_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, lw=3, color="#2563eb",
             label=f"模型 (AUC = {roc_auc_score(y_test, y_proba):.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="亂猜 (AUC = 0.5)")
axes[0].fill_between(fpr, tpr, alpha=.12, color="#2563eb")
axes[0].set_xlabel("FPR（健康的被誤抓比例）"); axes[0].set_ylabel("TPR = Recall")
axes[0].set_title("ROC 曲線"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(rec, prec, lw=3, color="#dc2626",
             label=f"AP = {average_precision_score(y_test, y_proba):.3f}")
axes[1].axhline(y_test.mean(), ls="--", color="gray",
                label=f"隨機猜的水準 = {y_test.mean():.3f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("PR 曲線（不平衡資料看這張比較誠實）")
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.show()

## 11. 特徵重要性：老闆最愛看的圖

In [ ]:
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
imp.plot.barh(ax=ax, color="#2563eb")
ax.set_title("特徵重要性（模型認為哪些欄位最有用）")
ax.set_xlabel("重要性")
plt.tight_layout()
plt.show()

print("👉 對照第 1 格造資料的規則，看模型有沒有找對重點。")

In [ ]:
# ⚠️ 樹模型內建的 feature_importances_ 會高估「取值很多」的欄位
# 更可靠的做法：Permutation Importance（把某欄打亂，看分數掉多少）
from sklearn.inspection import permutation_importance

r = permutation_importance(model, X_test, y_test, n_repeats=10,
                           random_state=RANDOM_STATE, scoring="f1", n_jobs=-1)

perm = pd.Series(r.importances_mean, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
perm.plot.barh(ax=ax, color="#16a34a", xerr=r.importances_std[perm.index.map(
    lambda c: list(X.columns).index(c))])
ax.set_title("Permutation Importance（比較可靠，面試講這個加分）")
ax.set_xlabel("打亂該欄後 F1 掉了多少")
plt.tight_layout()
plt.show()

## 12. 交叉驗證：確認結果不是運氣

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(model, X, y, cv=cv, scoring="f1", n_jobs=-1)

print("每一折的 F1：", np.round(scores, 3))
print(f"平均 F1: {scores.mean():.3f} ± {scores.std():.3f}")
print()
print("👉 報告成績一定要同時給平均和標準差。")
print("   標準差很大 = 模型不穩定，換一批資料表現就會變。")

---

## 🎯 動手改改看（重要，不要跳過）

改完之後重跑，觀察結果怎麼變：

1. **把 `class_weight="balanced"` 拿掉**，重跑第 7、8 格。
   → Recall 會掉多少？Precision 會升多少？想想為什麼。

2. **把 `max_depth` 從 10 改成 `None`**（不限制深度）。
   → 訓練集分數會變高還是低？測試集呢？這是什麼現象？

3. **把隨機森林換成 `LogisticRegression`**。
   → 記得邏輯迴歸需要先做 `StandardScaler`（樹模型不用）。分數差多少？

4. **把「累計消費」這個特徵拿掉**，看分數掉多少。
   → 這就是最土法煉鋼的特徵重要性驗證法。

5. **調整第 9 格的商業假設**（挽回成本、客戶價值、挽回成功率），
   → 看最佳閾值怎麼移動。這才是實務上真正在做的決策。

## 📝 下一步

- 把 `make_churn_data()` 換成 `pd.read_csv()` 讀真實資料
  （Kaggle 上搜 "Telco Customer Churn" 有一份非常經典的）
- 完成後接著做 `02_手刻神經網路.ipynb`